# ProcureSight AI — Compliance Risk Modelling

## 1. Prediction Task

Estimate the probability that a purchase order will receive the
dataset's non-compliant label.

Prediction point: after order details and negotiated pricing are
finalized, but before delivery and inspection outcomes are known.

Target:
- 1 = non-compliant
- 0 = compliant

This is an exploratory prediction task. The dataset does not document
the exact compliance rules or when labels became available, so it
cannot yet establish operational forecasting performance.

## 2. Candidate Features and Leakage Prevention

| Field | Initial decision | Reason |
|---|---|---|
| Supplier | Candidate input | Assumed known at order assessment |
| Item_Category | Candidate input | Describes the purchase |
| Quantity | Candidate input | Assumed finalized at assessment |
| Unit_Price | Candidate input | Assumed available at assessment |
| Negotiated_Price | Candidate input | Requires assessment after negotiation |
| Order_Date | Split and context | Preserves chronological evaluation |
| PO_ID | Exclude from inputs | Identifier retained for traceability |
| Delivery_Date | Exclude | Availability and meaning are uncertain |
| Order_Status | Exclude | May describe a later outcome |
| Defective_Units | Exclude | Obtained after inspection |
| Compliance / Non_Compliant | Target only | The outcome being predicted |

Exclude delivery intervals, defect-related missingness flags, and
full-dataset supplier summary metrics from the initial model.

Historical supplier aggregates would require evidence that their
underlying outcomes were available before each prediction date.

## 3. Evaluation Principles

- Compare models with a simple baseline.
- Use earlier orders for development and later orders for evaluation.
- Fit preprocessing only on training data.
- Select models and decision thresholds using development data.
- Report non-compliance precision and recall, alongside probability
  quality and the number of evaluated orders.
- Do not rely on accuracy alone: predicting every order as compliant
  would achieve 82.37% accuracy while detecting no non-compliant orders.
- Document that the full dataset has already been explored during
  descriptive analysis; the later evaluation period is not wholly unseen.

## 4. Chronological Data Split

- Training: orders before 1 January 2023.
- Validation: orders from 1 January through 30 June 2023.
- Test: orders from 1 July 2023 onward.

Use fixed calendar boundaries rather than randomly mixing past
and future orders. Keep the test set out of model and threshold selection.

The split follows order dates. Label-availability dates are unknown,
so this remains an exploratory temporal evaluation rather than a
verified simulation of operational forecasting.

The small validation and test samples will limit metric precision.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

model_df = pd.read_parquet(
    PROJECT_ROOT / "data" / "processed"
    / "procurement_prepared.parquet"
)

# Check the fields required for splitting and evaluation.
assert model_df["Order_Date"].notna().all()
assert model_df["Non_Compliant"].notna().all()
assert model_df["Non_Compliant"].isin([0, 1]).all()
assert model_df["PO_ID"].is_unique

model_df = model_df.sort_values(
    ["Order_Date", "PO_ID"]
).reset_index(drop=True)

VALIDATION_START = pd.Timestamp("2023-01-01")
TEST_START = pd.Timestamp("2023-07-01")

train_df = model_df.loc[
    model_df["Order_Date"] < VALIDATION_START
].copy()

validation_df = model_df.loc[
    (model_df["Order_Date"] >= VALIDATION_START)
    & (model_df["Order_Date"] < TEST_START)
].copy()

test_df = model_df.loc[
    model_df["Order_Date"] >= TEST_START
].copy()

# Confirm complete allocation and chronological separation.
assert len(train_df) + len(validation_df) + len(test_df) == len(model_df)
assert not any(
    frame.empty for frame in [train_df, validation_df, test_df]
)
assert train_df["Order_Date"].max() < validation_df["Order_Date"].min()
assert validation_df["Order_Date"].max() < test_df["Order_Date"].min()

split_summary = pd.DataFrame([
    {
        "Split": name,
        "Orders": len(frame),
        "First_Order": frame["Order_Date"].min().date(),
        "Last_Order": frame["Order_Date"].max().date(),
    }
    for name, frame in [
        ("Training", train_df),
        ("Validation", validation_df),
        ("Test", test_df),
    ]
])

display(split_summary)

,Split,Orders,First_Order,Last_Order
0,Training,396,2022-01-01,2022-12-30
1,Validation,176,2023-01-02,2023-06-30
2,Test,205,2023-07-01,2024-01-01


## 5. Input Features and Baseline

Use five candidate input fields:
Supplier, Item_Category, Quantity, Unit_Price, and Negotiated_Price.

The baseline assigns every order the training-set non-compliance
frequency as its predicted probability. It does not use order features.

At a probability threshold of 0.50, probabilities at or above the
threshold are classified as non-compliant.

Evaluate on validation data only at this stage. Keep the test set
reserved for final evaluation.

Report the confusion matrix alongside accuracy, precision, recall,
F1, and Brier score. Accuracy alone can hide missed non-compliance.

In [2]:
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    confusion_matrix,
)

FEATURES = [
    "Supplier",
    "Item_Category",
    "Quantity",
    "Unit_Price",
    "Negotiated_Price",
]

X_train = train_df[FEATURES].copy()
y_train = train_df["Non_Compliant"].astype(int)

X_validation = validation_df[FEATURES].copy()
y_validation = validation_df["Non_Compliant"].astype(int)

assert set(y_train.unique()) == {0, 1}

# Learn only the training-set class proportions.
baseline = DummyClassifier(strategy="prior")
baseline.fit(X_train, y_train)

positive_class_index = np.flatnonzero(baseline.classes_ == 1)[0]

baseline_probability = baseline.predict_proba(
    X_validation
)[:, positive_class_index]

baseline_prediction = (baseline_probability >= 0.50).astype(int)

baseline_results = pd.DataFrame([{
    "Model": "Training-frequency baseline",
    "Accuracy": accuracy_score(y_validation, baseline_prediction),
    "Precision": precision_score(
        y_validation, baseline_prediction, zero_division=0
    ),
    "Recall": recall_score(
        y_validation, baseline_prediction, zero_division=0
    ),
    "F1": f1_score(
        y_validation, baseline_prediction, zero_division=0
    ),
    "Brier_Score": brier_score_loss(
        y_validation, baseline_probability
    ),
}])

print(f"Training non-compliance rate: {y_train.mean():.2%}")
print(f"Validation orders: {len(y_validation)}")
print(f"Validation non-compliant orders: {y_validation.sum()}")

display(baseline_results.round(4))

baseline_confusion = pd.DataFrame(
    confusion_matrix(
        y_validation,
        baseline_prediction,
        labels=[0, 1],
    ),
    index=["Actual compliant", "Actual non-compliant"],
    columns=["Predicted compliant", "Predicted non-compliant"],
)

display(baseline_confusion)

Training non-compliance rate: 16.92%
Validation orders: 176
Validation non-compliant orders: 29


,Model,Accuracy,Precision,Recall,F1,Brier_Score
0,Training-frequency baseline,0.8352,0.0,0.0,0.0,0.1376


,Predicted compliant,Predicted non-compliant
Actual compliant,147,0
Actual non-compliant,29,0


### Split Proportions and Cross-Validation

The calendar-based split produces:

- Training: 396 orders (50.97%).
- Validation: 176 orders (22.65%).
- Test: 205 orders (26.38%).

This is not an 80/20 split. Calendar boundaries were selected to
preserve chronological order rather than achieve fixed percentages.

Five-fold cross-validation is not used at this stage.
Any later cross-validation must respect time order and remain
within the development data, leaving the final test period reserved.

### Baseline Confusion Matrix — Validation Results

A confusion matrix compares actual labels with predicted labels.

For this project:
- Positive (1) = non-compliant.
- Negative (0) = compliant.
- True = correct prediction; false = incorrect prediction.

| Actual / Predicted | Compliant (0) | Non-compliant (1) |
|---|---|---|
| Compliant (0) | True Negative (TN): 147 | False Positive (FP): 0 |
| Non-compliant (1) | False Negative (FN): 29 | True Positive (TP): 0 |

#### Business Meaning

- TN: a compliant order correctly classified as compliant.
- FP: a compliant order incorrectly flagged for review—a false alarm.
- FN: a non-compliant order incorrectly cleared—a missed risk.
- TP: a non-compliant order correctly flagged for review.

#### Interpretation

The baseline assigns every order the training non-compliance
frequency of 16.92%. At a 50% threshold, all 176 validation orders
are predicted compliant.

- Accuracy = (TP + TN) / all orders = 147/176 = 83.52%.
- Recall = TP / (TP + FN) = 0/29 = 0%.
- Precision = TP / (TP + FP). No orders were flagged, so its
  denominator is zero; our code reports 0 by convention.
- F1 is 0 because no positive cases were detected.
- Brier score is 0.1376 and evaluates probabilities, rather than
  the threshold-based classifications. Lower is better.

The baseline demonstrates why accuracy alone is insufficient:
it correctly classifies the majority class but detects none of
the non-compliant orders.

Future models should improve risk detection while measuring the
false alarms and review workload this creates.

### Model Used and Evaluation Stage

The current model is scikit-learn's
DummyClassifier(strategy="prior"), a training-frequency baseline.

It was fitted using 396 training orders through:
baseline.fit(X_train, y_train)

The fitted baseline learned the training non-compliance frequency
of 16.92%. It ignores feature values and assigns this same probability
to every order.

Validation probabilities were generated for 176 orders through:
baseline.predict_proba(X_validation)

A fixed initial threshold of 0.50 converted those probabilities
into class predictions. Metrics compared these predictions with
the actual validation labels, y_validation.

No candidate-model comparison or threshold optimization has
been performed yet. The 205 test orders have not been scored.

### Planned Model Comparison

Train candidate models on the training data and compare their
performance on the same validation period.

Fit all learned preprocessing on training data only.

Use validation results to assess risk detection, false alarms,
and probability quality, and to select a decision threshold.

Once the modelling choices are fixed, evaluate the selected
approach on the reserved test period. Do not use test results
to repeatedly adjust the model or threshold.

## 6. First Feature-Based Model: Logistic Regression

### Why This Model?

Logistic Regression learns relationships between order features and
the non-compliance label. It is a simple reference model to compare
with the training-frequency baseline and later, more flexible models.

### Preprocessing

- One-hot encode Supplier and Item_Category into indicator columns.
  This avoids assigning an artificial numeric ranking to categories.
- Standardize Quantity, Unit_Price, and Negotiated_Price.
- Fit preprocessing and the classifier together on training data only.
- Apply the fitted pipeline to validation data without refitting it.

### Initial Evaluation

Use the same five features, validation records, and 0.50 threshold
as the baseline. Start without class weighting to establish an
unweighted reference.

The test set remains reserved. Improved accuracy does not establish
fairness: missed risks and false alarms must also be examined by
supplier and category, with attention to small sample sizes.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

categorical_features = ["Supplier", "Item_Category"]
numeric_features = ["Quantity", "Unit_Price", "Negotiated_Price"]

# The audited candidate features are complete.
# Stop if that assumption no longer holds.
assert X_train[FEATURES].notna().all().all()
assert X_validation[FEATURES].notna().all().all()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categories",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        (
            "numbers",
            StandardScaler(),
            numeric_features,
        ),
    ],
    remainder="drop",
)

logistic_model = Pipeline([
    ("preprocessing", preprocessor),
    (
        "classifier",
        LogisticRegression(
            C=1.0,
            solver="lbfgs",
            max_iter=2000,
            class_weight=None,
        ),
    ),
])

# Learn preprocessing parameters and model coefficients on training only.
logistic_model.fit(X_train, y_train)

# Predict on validation data using the fitted pipeline.
positive_index = np.flatnonzero(logistic_model.classes_ == 1)[0]

logistic_probability = logistic_model.predict_proba(
    X_validation
)[:, positive_index]

logistic_prediction = (logistic_probability >= 0.50).astype(int)

logistic_results = pd.DataFrame([{
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_validation, logistic_prediction),
    "Precision": precision_score(
        y_validation, logistic_prediction, zero_division=0
    ),
    "Recall": recall_score(
        y_validation, logistic_prediction, zero_division=0
    ),
    "F1": f1_score(
        y_validation, logistic_prediction, zero_division=0
    ),
    "Brier_Score": brier_score_loss(
        y_validation, logistic_probability
    ),
}])

comparison_results = pd.concat(
    [baseline_results, logistic_results],
    ignore_index=True,
)
display(comparison_results.round(4))

logistic_confusion = pd.DataFrame(
    confusion_matrix(
        y_validation, logistic_prediction, labels=[0, 1]
    ),
    index=["Actual compliant", "Actual non-compliant"],
    columns=["Predicted compliant", "Predicted non-compliant"],
)
display(logistic_confusion)

,Model,Accuracy,Precision,Recall,F1,Brier_Score
0,Training-frequency baseline,0.8352,0.0,0.0,0.0,0.1376
1,Logistic Regression,0.8182,0.0,0.0,0.0,0.1264


,Predicted compliant,Predicted non-compliant
Actual compliant,144,3
Actual non-compliant,29,0


### Logistic Regression — Initial Validation Findings

At the fixed 0.50 threshold, Logistic Regression produces:
- TN = 144: correctly cleared compliant orders.
- FP = 3: compliant orders incorrectly flagged.
- FN = 29: missed non-compliant orders.
- TP = 0: correctly detected non-compliant orders.

Accuracy falls from 83.52% to 81.82%. Precision, recall, and F1
remain zero. At this threshold, the model provides no improvement
in risk detection and introduces three false alarms.

The Brier score improves from 0.1376 to 0.1264, indicating lower
average squared probability error on this validation sample.
This alone does not establish good calibration or useful ranking.

Further evaluation will examine probability ranking and the trade-off
between detected risks and false alarms across validation thresholds.
The test set remains reserved.

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Baseline", "Logistic Regression"],
    horizontal_spacing=0.16,
)

cell_names = [["TN", "FP"], ["FN", "TP"]]

for column, matrix in enumerate(
    [baseline_confusion, logistic_confusion],
    start=1,
):
    counts = matrix.to_numpy()
    labels = [
        [
            f"{cell_names[row][col]}: {counts[row, col]}"
            for col in range(2)
        ]
        for row in range(2)
    ]

    fig.add_trace(
        go.Heatmap(
            z=counts,
            x=["Compliant", "Non-compliant"],
            y=["Compliant", "Non-compliant"],
            text=labels,
            texttemplate="%{text}",
            coloraxis="coloraxis",
            hovertemplate=(
                "Actual: %{y}<br>Predicted: %{x}"
                "<br>Orders: %{z}<extra></extra>"
            ),
        ),
        row=1,
        col=column,
    )

fig.update_layout(
    title="Validation Confusion Matrices — Threshold 0.50",
    template="plotly_white",
    height=420,
    coloraxis=dict(
        colorscale="Blues",
        cmin=0,
        cmax=max(
            baseline_confusion.to_numpy().max(),
            logistic_confusion.to_numpy().max(),
        ),
        colorbar=dict(title="Orders"),
    ),
)

fig.update_xaxes(title_text="Predicted label")
fig.update_yaxes(title_text="Actual label", autorange="reversed")
fig.show()

## 7. Validation Threshold Analysis

Keep the fitted Logistic Regression model and its predicted
probabilities unchanged.

Compare several decision thresholds using validation data only.
An order is flagged when its predicted probability is greater
than or equal to the threshold.

For each threshold, report:
- TP: detected non-compliant orders.
- FN: missed non-compliant orders.
- FP: false alarms.
- TN: correctly cleared compliant orders.
- Precision, recall, and F1.
- Total flagged orders and review workload.

Lower thresholds flag at least as many orders, but precision and
F1 do not necessarily improve.

This is an exploratory comparison, not a final threshold selection.
No business review capacity or error costs have been established.
The test set remains reserved.

In [5]:
threshold_rows = []

for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    predictions = (
        logistic_probability >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        predictions,
        labels=[0, 1],
    ).ravel()

    threshold_rows.append({
        "Threshold": threshold,
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "TN": tn,
        "Precision": precision_score(
            y_validation, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_validation, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_validation, predictions, zero_division=0
        ),
        "Flagged_Orders": tp + fp,
        "Review_Workload_%": (
            100 * (tp + fp) / len(y_validation)
        ),
    })

threshold_summary = pd.DataFrame(threshold_rows)

display(threshold_summary.round(3))

,Threshold,TP,FN,FP,TN,Precision,Recall,F1,Flagged_Orders,Review_Workload_%
0,0.10,25,4,68,79,0.269,0.862,0.410,93,52.841
1,0.15,22,7,60,87,0.268,0.759,0.396,82,46.591
2,0.20,18,11,42,105,0.300,0.621,0.404,60,34.091
3,0.25,16,13,34,113,0.320,0.552,0.405,50,28.409
4,0.30,11,18,21,126,0.344,0.379,0.361,32,18.182
5,0.40,8,21,13,134,0.381,0.276,0.320,21,11.932
6,0.50,0,29,3,144,0.000,0.000,0.000,3,1.705


### Threshold Findings

Lowering the decision threshold improves risk detection on this
validation sample but increases false alarms and review workload.

- At 0.10, the model detects 25/29 non-compliant orders but flags
  93/176 orders, including 68 false alarms.
- At 0.25, it detects 16/29 and flags 50 orders, including 34 false
  alarms. Precision is 32% and recall is 55.2%.
- At 0.40, only 21 orders are flagged, but 21/29 non-compliant
  orders are missed.
- At 0.50, no non-compliant orders are detected.

The highest observed F1 among the tested thresholds is approximately
0.410 at 0.10. This is not automatically the best business decision,
and its small advantage over other thresholds may not generalize.

No final threshold has been selected. Business review capacity and
error costs remain unspecified. These results do not establish
fairness across suppliers or categories.

Changing the threshold does not retrain the model or change its
probability predictions. Its Brier score remains unchanged.

In [6]:
import plotly.express as px

threshold_plot = threshold_summary.copy()
threshold_plot["Recall_%"] = 100 * threshold_plot["Recall"]
threshold_plot["Precision_%"] = 100 * threshold_plot["Precision"]
threshold_plot["Threshold_Label"] = threshold_plot["Threshold"].map(
    lambda value: f"{value:.0%}"
)
threshold_plot = threshold_plot.sort_values("Review_Workload_%")

fig = px.line(
    threshold_plot,
    x="Review_Workload_%",
    y="Recall_%",
    markers=True,
    text="Threshold_Label",
    hover_data={
        "Threshold": ":.2f",
        "Precision_%": ":.1f",
        "TP": True,
        "FN": True,
        "FP": True,
        "Flagged_Orders": True,
        "Threshold_Label": False,
    },
    labels={
        "Review_Workload_%": "Orders flagged for review (%)",
        "Recall_%": "Non-compliant orders detected (%)",
        "Precision_%": "Precision (%)",
    },
    title="Validation: Risk Detection vs Review Workload",
    template="plotly_white",
)

fig.update_traces(
    textposition="top center",
    line_color="#2563EB",
    marker=dict(size=9),
)
fig.update_layout(height=450)
fig.update_xaxes(range=[0, 100], ticksuffix="%")
fig.update_yaxes(range=[0, 100], ticksuffix="%")

fig.show()

## 8. Validation Ranking Performance

Evaluate the predicted probabilities without selecting one
decision threshold.

- ROC-AUC measures discrimination between the two classes.
- Average Precision (AP) summarizes precision across recall levels.
  It is not the same calculation as trapezoidal area under a
  precision-recall curve.

Compare both measures against the constant-probability baseline
on the same validation records.

For this baseline, ROC-AUC is 0.5 and AP equals the validation
non-compliance prevalence: 29/176, approximately 0.1648.

Ranking performance does not establish probability calibration,
fairness, or an acceptable business review workload.
The test set remains reserved.

In [7]:
from sklearn.metrics import roc_auc_score, average_precision_score

assert y_validation.nunique() == 2

ranking_rows = []

for model_name, probabilities in [
    ("Training-frequency baseline", baseline_probability),
    ("Logistic Regression", logistic_probability),
]:
    ranking_rows.append({
        "Model": model_name,
        "ROC_AUC": roc_auc_score(y_validation, probabilities),
        "Average_Precision": average_precision_score(
            y_validation, probabilities
        ),
    })

ranking_results = pd.DataFrame(ranking_rows)

print(
    "Validation non-compliance prevalence: "
    f"{y_validation.mean():.2%}"
)
display(ranking_results.round(4))

Validation non-compliance prevalence: 16.48%


,Model,ROC_AUC,Average_Precision
0,Training-frequency baseline,0.5000,0.1648
1,Logistic Regression,0.7312,0.3170


### Ranking Findings

Logistic Regression achieves validation ROC-AUC of 0.7312 and
Average Precision of 0.3170, compared with 0.5000 and 0.1648
for the training-frequency baseline.

The model shows better discrimination than the constant baseline
on this sample, despite detecting no non-compliant orders at the
initial 0.50 threshold.

Ranking, probability quality, and threshold-based decisions measure
different aspects of performance. Improved ranking does not establish
calibration, fairness, or operational suitability.

These results are based on 176 validation orders, including only
29 non-compliant orders. No final model or threshold is selected.

In [8]:
from sklearn.metrics import precision_recall_curve
import plotly.graph_objects as go

pr_precision, pr_recall, _ = precision_recall_curve(
    y_validation,
    logistic_probability,
)

logistic_ap = ranking_results.loc[
    ranking_results["Model"] == "Logistic Regression",
    "Average_Precision",
].iloc[0]

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=pr_recall,
        y=pr_precision,
        mode="lines",
        name=f"Logistic Regression — AP {logistic_ap:.3f}",
        line=dict(color="#2563EB", shape="hv"),
        hovertemplate=(
            "Recall: %{x:.1%}<br>"
            "Precision: %{y:.1%}<extra></extra>"
        ),
    )
)

fig.add_hline(
    y=y_validation.mean(),
    line_dash="dash",
    line_color="#64748B",
    annotation_text=(
        f"Prevalence reference: {y_validation.mean():.2%}"
    ),
    annotation_position="bottom right",
)

fig.update_layout(
    title="Validation Precision–Recall Curve",
    template="plotly_white",
    height=450,
    xaxis_title="Recall — share of non-compliant orders detected",
    yaxis_title="Precision — share of flagged orders non-compliant",
    legend=dict(orientation="h", y=-0.25),
)

fig.update_xaxes(range=[0, 1], tickformat=".0%")
fig.update_yaxes(range=[0, 1.05], tickformat=".0%")

fig.show()

## 9. Explainable Boosting Machine — Next Candidate

### Why EBM?

An Explainable Boosting Machine can learn nonlinear relationships
while exposing the feature contributions behind its predictions.

We will begin with an additive model without feature interactions,
so each feature's learned contribution can be examined separately.

### Comparison Plan

- Use the same five input features and chronological split.
- Fit the model on training data only.
- Evaluate on the same validation records as the earlier models.
- Compare ranking, probability quality, and threshold-based results.
- Keep the test period reserved.

### Interpretation Limits

Feature contributions explain the model's calculations, not causal
effects in the real world.

An interpretable model is not automatically accurate or fair.
EBM must demonstrate its usefulness through evaluation.

### Initial EBM Configuration

- interactions=0: learn separate feature effects without interactions.
- max_bins=32: limit numeric feature binning for this small dataset.
- min_samples_leaf=10: require more observations in tree leaves.
- learning_rate=0.02: use small boosting updates.
- max_rounds=1000: set a fixed initial training budget.
- validation_size=0: disable internal validation and early stopping.
- outer_bags=1: use a single fit without outer-bag averaging.
- random_state=42: make the randomized operations reproducible.
- n_jobs=1: limit parallel processing for this small experiment.

These are initial settings, not optimized parameters.
This configuration does not provide bag-based uncertainty estimates.

Fit on the 396 training orders and evaluate on the same 176
validation orders used previously. Keep the 205 test orders reserved.

In [9]:
from interpret.glassbox import ExplainableBoostingClassifier
from time import perf_counter

ebm_model = ExplainableBoostingClassifier(
    feature_names=FEATURES,
    feature_types=[
        "nominal",     # Supplier
        "nominal",     # Item_Category
        "continuous",  # Quantity
        "continuous",  # Unit_Price
        "continuous",  # Negotiated_Price
    ],
    interactions=0,
    max_bins=32,
    min_samples_leaf=10,
    learning_rate=0.02,
    max_rounds=1000,
    validation_size=0,
    outer_bags=1,
    random_state=42,
    n_jobs=1,
)

start_time = perf_counter()
print("Training EBM...", flush=True)

# Use the original five feature columns, not encoded/scaled inputs.
ebm_model.fit(X_train[FEATURES], y_train)

print(f"Training completed in {perf_counter() - start_time:.1f} seconds.")

positive_index = np.flatnonzero(ebm_model.classes_ == 1)[0]

ebm_probability = ebm_model.predict_proba(
    X_validation[FEATURES]
)[:, positive_index]

ebm_prediction = (ebm_probability >= 0.50).astype(int)

# Compare all models on the same validation records and threshold.
model_probabilities = {
    "Training-frequency baseline": baseline_probability,
    "Logistic Regression": logistic_probability,
    "EBM — additive": ebm_probability,
}

comparison_rows = []

for name, probabilities in model_probabilities.items():
    predictions = (probabilities >= 0.50).astype(int)

    comparison_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_validation, predictions),
        "Precision": precision_score(
            y_validation, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_validation, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_validation, predictions, zero_division=0
        ),
        "Brier_Score": brier_score_loss(
            y_validation, probabilities
        ),
        "ROC_AUC": roc_auc_score(y_validation, probabilities),
        "Average_Precision": average_precision_score(
            y_validation, probabilities
        ),
    })

comparison_results = pd.DataFrame(comparison_rows)
display(comparison_results.round(4))

ebm_confusion = pd.DataFrame(
    confusion_matrix(y_validation, ebm_prediction, labels=[0, 1]),
    index=["Actual compliant", "Actual non-compliant"],
    columns=["Predicted compliant", "Predicted non-compliant"],
)
display(ebm_confusion)

Training EBM...
Training completed in 0.1 seconds.


,Model,Accuracy,Precision,Recall,F1,Brier_Score,ROC_AUC,Average_Precision
0,Training-frequency baseline,0.8352,0.0,0.000,0.0000,0.1376,0.5000,0.1648
1,Logistic Regression,0.8182,0.0,0.000,0.0000,0.1264,0.7312,0.3170
2,EBM — additive,0.8011,0.2,0.069,0.1026,0.1372,0.7117,0.2938


,Predicted compliant,Predicted non-compliant
Actual compliant,139,8
Actual non-compliant,27,2


### Initial EBM Validation Findings

The additive EBM completed training successfully.

At threshold 0.50:
- TP = 2 and FN = 27: recall is 6.9%.
- FP = 8 and TN = 139: precision is 20%.
- Ten orders are flagged for review.

Although EBM detects two non-compliant orders at this threshold,
Logistic Regression has better validation ROC-AUC, Average Precision,
and Brier score.

The current EBM configuration has not demonstrated an overall
advantage. Its interpretability does not guarantee superior
predictive performance or fairness.

Compare both models across the same threshold grid before drawing
conclusions about their review workload and detection trade-offs.
No final model or threshold has been selected.

In [10]:
comparison_threshold_rows = []

for model_name, probabilities in [
    ("Logistic Regression", logistic_probability),
    ("EBM — additive", ebm_probability),
]:
    for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
        predictions = (probabilities >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_validation, predictions, labels=[0, 1]
        ).ravel()

        comparison_threshold_rows.append({
            "Model": model_name,
            "Threshold": threshold,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn,
            "Precision": precision_score(
                y_validation, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_validation, predictions, zero_division=0
            ),
            "F1": f1_score(
                y_validation, predictions, zero_division=0
            ),
            "Flagged_Orders": tp + fp,
            "Review_Workload_%": 100 * (tp + fp) / len(y_validation),
        })

model_threshold_comparison = pd.DataFrame(
    comparison_threshold_rows
).sort_values(["Threshold", "Model"]).reset_index(drop=True)

display(model_threshold_comparison.round(3))

,Model,Threshold,TP,FN,FP,TN,Precision,Recall,F1,Flagged_Orders,Review_Workload_%
0,EBM — additive,0.10,22,7,67,80,0.247,0.759,0.373,89,50.568
1,Logistic Regression,0.10,25,4,68,79,0.269,0.862,0.410,93,52.841
2,EBM — additive,0.15,19,10,53,94,0.264,0.655,0.376,72,40.909
3,Logistic Regression,0.15,22,7,60,87,0.268,0.759,0.396,82,46.591
4,EBM — additive,0.20,18,11,41,106,0.305,0.621,0.409,59,33.523
5,Logistic Regression,0.20,18,11,42,105,0.300,0.621,0.404,60,34.091
6,EBM — additive,0.25,14,15,33,114,0.298,0.483,0.368,47,26.705
7,Logistic Regression,0.25,16,13,34,113,0.320,0.552,0.405,50,28.409
8,EBM — additive,0.30,10,19,26,121,0.278,0.345,0.308,36,20.455
9,Logistic Regression,0.30,11,18,21,126,0.344,0.379,0.361,32,18.182


### Comparing Detection and Review Workload

At threshold 0.20, both models detect 18 of 29 non-compliant
orders. EBM raises 41 false alarms and Logistic Regression raises 42.

At threshold 0.30, Logistic Regression detects more risks
with fewer false alarms than EBM.

The highest F1 values among the tested settings are very close:
approximately 0.410 for Logistic Regression and 0.409 for EBM.
These values occur at different thresholds and workloads.

Logistic Regression remains stronger on validation ROC-AUC,
Average Precision, and Brier score. EBM has not demonstrated
a consistent predictive advantage in this initial experiment.

No final model or threshold is selected. The chart below connects
the tested operating points; intermediate points on the lines
are not separately evaluated policies.

In [11]:
import plotly.express as px

workload_comparison = model_threshold_comparison.copy()
workload_comparison["Recall_%"] = (
    100 * workload_comparison["Recall"]
)
workload_comparison["Precision_%"] = (
    100 * workload_comparison["Precision"]
)

workload_comparison = workload_comparison.sort_values(
    ["Model", "Review_Workload_%"]
)

fig = px.line(
    workload_comparison,
    x="Review_Workload_%",
    y="Recall_%",
    color="Model",
    markers=True,
    hover_data={
        "Threshold": ":.2f",
        "Review_Workload_%": ":.1f",
        "Recall_%": ":.1f",
        "Precision_%": ":.1f",
        "TP": True,
        "FP": True,
        "FN": True,
        "Flagged_Orders": True,
    },
    color_discrete_map={
        "Logistic Regression": "#2563EB",
        "EBM — additive": "#F97316",
    },
    labels={
        "Review_Workload_%": "Orders flagged for review (%)",
        "Recall_%": "Non-compliant orders detected (%)",
        "Precision_%": "Precision (%)",
    },
    title="Validation: Model Detection vs Review Workload",
    template="plotly_white",
)

fig.update_traces(marker=dict(size=8))
fig.update_xaxes(range=[0, 100], ticksuffix="%")
fig.update_yaxes(range=[0, 100], ticksuffix="%")
fig.update_layout(
    height=470,
    legend=dict(
        title_text="",
        orientation="h",
        y=-0.25,
    ),
)

fig.show()

## 10. EBM Explainability — Global Feature Importance

Global importance summarizes how strongly each feature contributes
to the fitted EBM's scores across the training data.

Because interactions are disabled, each model term corresponds
to one input feature.

### Interpretation

- A larger value means a larger average absolute contribution.
- Importance does not show whether a feature raises or lowers risk.
- Values are model-score contributions in log-odds units,
  not probabilities or percentage-point changes.
- Importance explains this fitted EBM, not Logistic Regression.
- It does not establish causation, fairness, or predictive value
  on unseen data.
- Related features, such as unit and negotiated prices, can share
  predictive information, affecting their individual importance.

In [12]:
import plotly.express as px

ebm_importance = pd.DataFrame({
    "Feature": ebm_model.term_names_,
    "Mean_Absolute_Contribution": ebm_model.term_importances(),
}).sort_values(
    "Mean_Absolute_Contribution",
    ascending=False,
).reset_index(drop=True)

display(ebm_importance.round(4))

fig = px.bar(
    ebm_importance,
    x="Mean_Absolute_Contribution",
    y="Feature",
    orientation="h",
    text="Mean_Absolute_Contribution",
    title="EBM Global Feature Importance — Training Distribution",
    labels={
        "Mean_Absolute_Contribution":
            "Mean absolute contribution (log-odds)",
    },
    template="plotly_white",
)

fig.update_traces(
    marker_color="#2563EB",
    texttemplate="%{text:.3f}",
    textposition="outside",
    cliponaxis=False,
)

fig.update_yaxes(
    categoryorder="array",
    categoryarray=ebm_importance["Feature"].tolist(),
    autorange="reversed",
    title_text=None,
)

fig.update_xaxes(
    range=[
        0,
        max(
            ebm_importance["Mean_Absolute_Contribution"].max() * 1.2,
            0.01,
        ),
    ]
)

fig.update_layout(height=400)
fig.show()

,Feature,Mean_Absolute_Contribution
0,Supplier,1.9348
1,Negotiated_Price,0.8062
2,Unit_Price,0.7804
3,Quantity,0.3365
4,Item_Category,0.3237


### Global Importance Findings

Supplier has the largest average absolute contribution (1.935),
followed by Negotiated_Price (0.806) and Unit_Price (0.780).

Quantity (0.336) and Item_Category (0.324) have smaller contributions
in this fitted model.

These values measure contribution magnitude in log-odds units.
They do not represent percentages, causal effects, or improvements
in predictive accuracy.

Strong reliance on Supplier warrants further investigation:
- Examine which supplier values raise or lower the model score.
- Review validation errors by supplier, including group counts.
- Later compare performance with and without Supplier.

Importance alone does not establish unfairness. It also does not
demonstrate that the model will generalize to previously unseen suppliers.

## 11. EBM Supplier Effects

Inspect the learned contribution for each supplier.

A positive contribution increases the non-compliance log-odds score;
a negative contribution decreases it. The final probability depends
on the intercept and all feature contributions together.

These effects describe the fitted model's associations, not the
causal impact of choosing a supplier.

Our single-fit configuration does not provide bag-based uncertainty
estimates.

In [13]:
# Generate the fitted EBM's built-in global explanation.
ebm_global = ebm_model.explain_global(
    name="ProcureSight — Additive EBM"
)

# Find the supplier term rather than assuming its position.
supplier_term_index = list(ebm_model.term_names_).index("Supplier")

# Display its learned categorical effects.
supplier_effect_fig = ebm_global.visualize(supplier_term_index)
supplier_effect_fig.show()

### Interpretation of the EBM Supplier Effects

The supplier-effect chart shows the direction of each supplier’s
contribution to the EBM’s predicted non-compliance risk.

- Delta_Logistics has the strongest positive contribution.
- Beta_Supplies and Gamma_Co also have positive contributions.
- Alpha_Inc has a slightly negative contribution.
- Epsilon_Group has the strongest negative contribution.

Scores are contributions in log-odds, not probabilities or percentages.
An order’s final probability combines the model intercept with all
feature contributions.

The lower density chart shows how many training observations support
each supplier category.

These effects describe associations learned from the training data.
They do not establish causation or demonstrate fairness. We will examine
validation errors by supplier, including missed non-compliant orders
and incorrectly flagged compliant orders.

## 12. Validation Errors by Supplier

We compare Logistic Regression and EBM at an exploratory threshold of
0.20 using the validation set.

- True Positive (TP): non-compliant order correctly flagged.
- False Negative (FN): non-compliant order missed.
- False Positive (FP): compliant order incorrectly flagged.
- True Negative (TN): compliant order correctly left unflagged.

Recall measures the percentage of non-compliant orders detected.
False-positive rate measures the percentage of compliant orders
incorrectly flagged.

Each rate is shown alongside its supporting counts. A rate is undefined
when its denominator is zero and is reported as missing, rather than 0%.

Differences between suppliers warrant investigation, but this small
dataset cannot establish fairness from these results alone.
The threshold is exploratory; the test set is not used.

In [14]:
import numpy as np
import pandas as pd

AUDIT_THRESHOLD = 0.20
audit_rows = []

# Use positional arrays to keep suppliers, labels, and predictions aligned.
validation_suppliers = X_validation["Supplier"].to_numpy()
actual = np.asarray(y_validation, dtype=int)

for model_name, probabilities in {
    "Logistic Regression": logistic_probability,
    "EBM": ebm_probability,
}.items():
    probabilities = np.asarray(probabilities)

    assert len(probabilities) == len(actual) == len(validation_suppliers)

    audit_data = pd.DataFrame({
        "Supplier": validation_suppliers,
        "Actual": actual,
        "Predicted": (probabilities >= AUDIT_THRESHOLD).astype(int),
    })

    for supplier, group in audit_data.groupby("Supplier", dropna=False):
        positive = group["Actual"].eq(1)
        flagged = group["Predicted"].eq(1)

        tp = int((positive & flagged).sum())
        fn = int((positive & ~flagged).sum())
        fp = int((~positive & flagged).sum())
        tn = int((~positive & ~flagged).sum())

        audit_rows.append({
            "Model": model_name,
            "Supplier": supplier,
            "Orders": len(group),
            "Actual_Non_Compliant": tp + fn,
            "Actual_Compliant": fp + tn,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn,
            "Recall_%": 100 * tp / (tp + fn) if tp + fn else np.nan,
            "False_Positive_Rate_%": (
                100 * fp / (fp + tn) if fp + tn else np.nan
            ),
        })

supplier_error_audit = (
    pd.DataFrame(audit_rows)
    .sort_values(["Supplier", "Model"])
    .reset_index(drop=True)
)

display(supplier_error_audit.round(2))

,Model,Supplier,Orders,Actual_Non_Compliant,Actual_Compliant,TP,FN,FP,TN,Recall_%,False_Positive_Rate_%
0,EBM,Alpha_Inc,29,3,26,0,3,0,26,0.00,0.00
1,Logistic Regression,Alpha_Inc,29,3,26,0,3,0,26,0.00,0.00
2,EBM,Beta_Supplies,34,7,27,5,2,14,13,71.43,51.85
3,Logistic Regression,Beta_Supplies,34,7,27,4,3,19,8,57.14,70.37
4,EBM,Delta_Logistics,37,14,23,13,1,18,5,92.86,78.26
5,Logistic Regression,Delta_Logistics,37,14,23,14,0,23,0,100.00,100.00
6,EBM,Epsilon_Group,43,1,42,0,1,0,42,0.00,0.00
7,Logistic Regression,Epsilon_Group,43,1,42,0,1,0,42,0.00,0.00
8,EBM,Gamma_Co,33,4,29,0,4,9,20,0.00,31.03
9,Logistic Regression,Gamma_Co,33,4,29,0,4,0,29,0.00,0.00


## 13. CatBoost Comparison

We train a CatBoost classifier using the same five order-time features
and chronological split as the previous models.

CatBoost uses boosted decision trees and handles Supplier and
Item_Category as categorical inputs. This provides a comparison with
Logistic Regression and our additive EBM.

This first experiment uses fixed settings:
- 300 boosting iterations
- Tree depth of 4
- Learning rate of 0.03
- Random seed of 42

The model is fitted only on the training set. We do not use validation
data for early stopping in this experiment.

Validation metrics are compared at threshold 0.50, alongside ROC-AUC,
average precision, and Brier score. The threshold is a reference point,
not a final operating decision. The test set remains reserved.

In [15]:
from time import perf_counter
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
)

catboost_model = CatBoostClassifier(
    iterations=300,
    depth=4,
    learning_rate=0.03,
    loss_function="Logloss",
    cat_features=["Supplier", "Item_Category"],
    random_seed=42,
    thread_count=1,
    verbose=False,
    allow_writing_files=False,
)

start = perf_counter()

# Fit using training data only.
catboost_model.fit(X_train[FEATURES], y_train)

print(f"Training completed in {perf_counter() - start:.2f} seconds")

positive_class_index = list(catboost_model.classes_).index(1)

catboost_probability = catboost_model.predict_proba(
    X_validation[FEATURES]
)[:, positive_class_index]

# Rebuild the comparison so rerunning this cell does not duplicate rows.
model_probabilities = {
    "Dummy baseline": baseline_probability,
    "Logistic Regression": logistic_probability,
    "EBM": ebm_probability,
    "CatBoost": catboost_probability,
}

comparison_rows = []

for model_name, probabilities in model_probabilities.items():
    prediction = (probabilities >= 0.50).astype(int)

    comparison_rows.append({
        "Model": model_name,
        "Threshold": 0.50,
        "Accuracy": accuracy_score(y_validation, prediction),
        "Precision": precision_score(
            y_validation, prediction, zero_division=0
        ),
        "Recall": recall_score(
            y_validation, prediction, zero_division=0
        ),
        "F1": f1_score(y_validation, prediction, zero_division=0),
        "ROC_AUC": roc_auc_score(y_validation, probabilities),
        "Average_Precision": average_precision_score(
            y_validation, probabilities
        ),
        "Brier_Score": brier_score_loss(y_validation, probabilities),
    })

comparison_results = pd.DataFrame(comparison_rows)
display(comparison_results.round(4))

catboost_prediction = (catboost_probability >= 0.50).astype(int)

catboost_confusion = pd.DataFrame(
    confusion_matrix(
        y_validation, catboost_prediction, labels=[0, 1]
    ),
    index=["Actual compliant", "Actual non-compliant"],
    columns=["Predicted compliant", "Predicted non-compliant"],
)

display(catboost_confusion)

Training completed in 4.05 seconds


,Model,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,Average_Precision,Brier_Score
0,Dummy baseline,0.5,0.8352,0.0,0.0000,0.0000,0.5000,0.1648,0.1376
1,Logistic Regression,0.5,0.8182,0.0,0.0000,0.0000,0.7312,0.3170,0.1264
2,EBM,0.5,0.8011,0.2,0.0690,0.1026,0.7117,0.2938,0.1372
3,CatBoost,0.5,0.8409,1.0,0.0345,0.0667,0.7579,0.3725,0.1233


,Predicted compliant,Predicted non-compliant
Actual compliant,147,0
Actual non-compliant,28,1


### CatBoost Validation Findings

CatBoost achieved the highest validation ROC-AUC (0.7579) and average
precision (0.3725), and the lowest Brier score (0.1233), among the models
evaluated so far.

At threshold 0.50, however, it detected only 1 of 29 non-compliant orders:
TN = 147, FP = 0, FN = 28, TP = 1.

Its 100% precision is based on just one flagged order. Its 84.09% accuracy
must be interpreted alongside its low recall of 3.45%.

CatBoost is a promising candidate for further comparison. We will
examine the trade-off between detected risks, false alarms, and review
workload at lower thresholds. No final model or threshold is selected,
and the test set remains reserved.

In [16]:
threshold_rows = []

for model_name, probabilities in {
    "Logistic Regression": logistic_probability,
    "EBM": ebm_probability,
    "CatBoost": catboost_probability,
}.items():
    for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
        prediction = (probabilities >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_validation, prediction, labels=[0, 1]
        ).ravel()

        threshold_rows.append({
            "Model": model_name,
            "Threshold": threshold,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn,
            "Precision_%": (
                100 * tp / (tp + fp) if tp + fp else np.nan
            ),
            "Recall_%": (
                100 * tp / (tp + fn) if tp + fn else np.nan
            ),
            "F1": f1_score(
                y_validation, prediction, zero_division=0
            ),
            "Flagged_Orders": int(prediction.sum()),
            "Review_Workload_%": 100 * prediction.mean(),
        })

model_threshold_comparison = (
    pd.DataFrame(threshold_rows)
    .sort_values(["Threshold", "Model"])
    .reset_index(drop=True)
)

display(model_threshold_comparison.round(2))

,Model,Threshold,TP,FN,FP,TN,Precision_%,Recall_%,F1,Flagged_Orders,Review_Workload_%
0,CatBoost,0.10,23,6,65,82,26.14,79.31,0.39,88,50.00
1,EBM,0.10,22,7,67,80,24.72,75.86,0.37,89,50.57
2,Logistic Regression,0.10,25,4,68,79,26.88,86.21,0.41,93,52.84
3,CatBoost,0.15,21,8,48,99,30.43,72.41,0.43,69,39.20
4,EBM,0.15,19,10,53,94,26.39,65.52,0.38,72,40.91
5,Logistic Regression,0.15,22,7,60,87,26.83,75.86,0.40,82,46.59
6,CatBoost,0.20,15,14,32,115,31.91,51.72,0.39,47,26.70
7,EBM,0.20,18,11,41,106,30.51,62.07,0.41,59,33.52
8,Logistic Regression,0.20,18,11,42,105,30.00,62.07,0.40,60,34.09
9,CatBoost,0.25,11,18,19,128,36.67,37.93,0.37,30,17.05


### Validation Trade-off: Risk Detection and Review Workload

Lower thresholds generally detect more non-compliant orders while
flagging more compliant orders for review.

At threshold 0.15, CatBoost detects 21 of 29 non-compliant orders and
flags 69 orders in total. Logistic Regression detects 22 but flags 82;
EBM detects 19 and flags 72.

The same threshold does not produce the same review workload across
models. We therefore compare recall against the percentage of orders
flagged, alongside false-alarm counts.

Each plotted point represents an evaluated threshold. These validation
results do not select a final operating threshold or establish fairness.
The test set remains reserved.

In [17]:
import plotly.express as px

workload_fig = px.scatter(
    model_threshold_comparison,
    x="Review_Workload_%",
    y="Recall_%",
    color="Model",
    symbol="Model",
    hover_data={
        "Threshold": ":.2f",
        "TP": True,
        "FN": True,
        "FP": True,
        "Flagged_Orders": True,
        "Precision_%": ":.2f",
        "Review_Workload_%": ":.2f",
        "Recall_%": ":.2f",
    },
    title="Validation: Risk Detection versus Review Workload",
    labels={
        "Review_Workload_%": "Orders flagged for review (%)",
        "Recall_%": "Non-compliant orders detected (%)",
    },
    color_discrete_map={
        "Logistic Regression": "#2563EB",
        "EBM": "#F97316",
        "CatBoost": "#16A34A",
    },
)

workload_fig.update_traces(marker_size=11)
workload_fig.update_xaxes(range=[0, 60], ticksuffix="%")
workload_fig.update_yaxes(range=[0, 100], ticksuffix="%")
workload_fig.update_layout(
    template="plotly_white",
    height=500,
    legend_title_text="Model",
)

workload_fig.show()

## 14. Feature Engineering Experiment

We test two features calculated from information available when an
order is assessed:

- Negotiated_Order_Value = Quantity × Negotiated_Price
- Discount_Percent = 100 × (Unit_Price − Negotiated_Price) / Unit_Price

These features express order size and relative negotiated savings.
They do not introduce new source information or guarantee improvement.

We retain the original five inputs and add these two features.
The original models remain available for comparison.

The same chronological training and validation sets are used.
The test set remains reserved. No currency is assumed because the
dataset does not identify one.

In [18]:
def add_order_features(frame):
    """Create order-time features without changing the original data."""
    result = frame.copy()

    assert result["Unit_Price"].gt(0).all(), (
        "Unit prices must be positive to calculate discount percentage."
    )

    result["Negotiated_Order_Value"] = (
        result["Quantity"] * result["Negotiated_Price"]
    )

    result["Discount_Percent"] = (
        100
        * (result["Unit_Price"] - result["Negotiated_Price"])
        / result["Unit_Price"]
    )

    return result


X_train_engineered = add_order_features(X_train[FEATURES])
X_validation_engineered = add_order_features(X_validation[FEATURES])

ENGINEERED_FEATURES = FEATURES + [
    "Negotiated_Order_Value",
    "Discount_Percent",
]

assert np.isfinite(
    X_train_engineered[
        ["Negotiated_Order_Value", "Discount_Percent"]
    ].to_numpy(dtype=float)
).all()

assert np.isfinite(
    X_validation_engineered[
        ["Negotiated_Order_Value", "Discount_Percent"]
    ].to_numpy(dtype=float)
).all()

print("Original features:", len(FEATURES))
print("Features after engineering:", len(ENGINEERED_FEATURES))
print("Training orders:", len(X_train_engineered))
print("Validation orders:", len(X_validation_engineered))

display(
    X_train_engineered[
        [
            "Quantity",
            "Unit_Price",
            "Negotiated_Price",
            "Negotiated_Order_Value",
            "Discount_Percent",
        ]
    ].head().round(2)
)

Original features: 5
Features after engineering: 7
Training orders: 396
Validation orders: 176


,Quantity,Unit_Price,Negotiated_Price,Negotiated_Order_Value,Discount_Percent
0,552,21.52,18.79,10372.08,12.69
1,1042,36.74,35.78,37282.76,2.61
2,85,22.54,20.68,1757.80,8.25
3,172,63.61,55.92,9618.24,12.09
4,456,80.53,77.86,35504.16,3.32


### Training Feature Distributions and Correlations

We inspect numeric features using the training set only.

Histograms show the distribution of the two engineered features.
A Pearson correlation heatmap shows linear relationships between
numeric inputs, ranging from −1 to +1.

The engineered features are calculated from existing inputs, so some
correlation is expected. Correlation does not establish causation or
determine whether a feature improves predictions.

Supplier and Item_Category are excluded from this numeric heatmap.
We do not automatically remove features because they are correlated.
Their usefulness will be assessed through the model comparison.

In [19]:
import plotly.express as px
from plotly.subplots import make_subplots

# Inspect the two new features on their original scales.
distribution_fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Negotiated Order Value",
        "Discount Percentage",
    ],
)

for column_number, feature in enumerate(
    ["Negotiated_Order_Value", "Discount_Percent"],
    start=1,
):
    histogram = px.histogram(
        X_train_engineered,
        x=feature,
        nbins=25,
    ).data[0]

    distribution_fig.add_trace(
        histogram,
        row=1,
        col=column_number,
    )

distribution_fig.update_layout(
    title="Engineered Feature Distributions — Training Set",
    template="plotly_white",
    showlegend=False,
    height=400,
)
distribution_fig.update_yaxes(title_text="Order count")
distribution_fig.update_xaxes(
    title_text="Order value (currency unspecified)",
    row=1, col=1,
)
distribution_fig.update_xaxes(
    title_text="Discount (%)",
    row=1, col=2,
)
distribution_fig.show()

# Pearson correlation between numeric inputs only.
numeric_features = [
    "Quantity",
    "Unit_Price",
    "Negotiated_Price",
    "Negotiated_Order_Value",
    "Discount_Percent",
]

training_correlations = (
    X_train_engineered[numeric_features].corr(method="pearson")
)

correlation_fig = px.imshow(
    training_correlations,
    text_auto=".2f",
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    aspect="auto",
    title="Numeric Feature Correlations — Training Set",
)

correlation_fig.update_layout(
    template="plotly_white",
    height=550,
    coloraxis_colorbar_title="Pearson r",
)
correlation_fig.show()

### Numeric Correlation Findings

Unit_Price and Negotiated_Price are strongly correlated (Pearson r ≈ 0.99).
Negotiated_Order_Value is positively correlated with Quantity (≈ 0.70)
and Negotiated_Price (≈ 0.59), consistent with its calculation.

Discount_Percent has weak linear correlations with the other numeric
inputs. This does not establish its predictive usefulness.

We retain all seven features for the planned experiment. We compare
CatBoost with and without the engineered features using identical
training settings and the same validation set.

In [20]:
# Reuse the original CatBoost settings, changing only the input features.
catboost_engineered_model = CatBoostClassifier(
    **catboost_model.get_params()
)

start = perf_counter()

catboost_engineered_model.fit(
    X_train_engineered[ENGINEERED_FEATURES],
    y_train,
)

print(f"Training completed in {perf_counter() - start:.2f} seconds")

positive_index = list(catboost_engineered_model.classes_).index(1)

catboost_engineered_probability = (
    catboost_engineered_model.predict_proba(
        X_validation_engineered[ENGINEERED_FEATURES]
    )[:, positive_index]
)

feature_comparison_rows = []

for experiment, probabilities in {
    "CatBoost — original 5 features": catboost_probability,
    "CatBoost — engineered 7 features": catboost_engineered_probability,
}.items():
    feature_comparison_rows.append({
        "Experiment": experiment,
        "ROC_AUC": roc_auc_score(y_validation, probabilities),
        "Average_Precision": average_precision_score(
            y_validation, probabilities
        ),
        "Brier_Score": brier_score_loss(
            y_validation, probabilities
        ),
    })

feature_comparison_results = pd.DataFrame(feature_comparison_rows)
display(feature_comparison_results.round(4))

# Compare both versions at the same exploratory threshold.
feature_threshold_rows = []

for experiment, probabilities in {
    "Original 5 features": catboost_probability,
    "Engineered 7 features": catboost_engineered_probability,
}.items():
    prediction = (probabilities >= 0.15).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation, prediction, labels=[0, 1]
    ).ravel()

    feature_threshold_rows.append({
        "Experiment": experiment,
        "Threshold": 0.15,
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "TN": tn,
        "Precision_%": 100 * tp / (tp + fp) if tp + fp else np.nan,
        "Recall_%": 100 * tp / (tp + fn) if tp + fn else np.nan,
        "Flagged_Orders": int(prediction.sum()),
        "Review_Workload_%": 100 * prediction.mean(),
    })

display(pd.DataFrame(feature_threshold_rows).round(2))

Training completed in 5.59 seconds


,Experiment,ROC_AUC,Average_Precision,Brier_Score
0,CatBoost — original 5 features,0.7579,0.3725,0.1233
1,CatBoost — engineered 7 features,0.7478,0.3981,0.1218


,Experiment,Threshold,TP,FN,FP,TN,Precision_%,Recall_%,Flagged_Orders,Review_Workload_%
0,Original 5 features,0.15,21,8,48,99,30.43,72.41,69,39.2
1,Engineered 7 features,0.15,21,8,48,99,30.43,72.41,69,39.2


### Feature Engineering Experiment Findings

Adding Negotiated_Order_Value and Discount_Percent produced mixed
validation results:

- Average precision increased from 0.3725 to 0.3981.
- Brier score decreased from 0.1233 to 0.1218.
- ROC-AUC decreased from 0.7579 to 0.7478.

At threshold 0.15, both versions produced the same confusion-matrix
counts: TP = 21, FN = 8, FP = 48, TN = 99. Both flagged 69 orders.
Identical counts do not establish that the same orders were flagged.

The engineered features show no clear overall advantage on this
validation set. We retain both experiments and use the original
five-feature CatBoost as the provisional reference for subsequent
checks because it is simpler.

No final model or operating threshold has been selected.
The test set remains reserved.

## 15. CatBoost Validation Errors by Supplier

We audit the original five-feature CatBoost at the exploratory
threshold of 0.15.

For each supplier, we report the number of compliant and non-compliant
orders, confusion-matrix counts, recall, and false-positive rate.

Recall uses actual non-compliant orders as its denominator.
False-positive rate uses actual compliant orders as its denominator.
Undefined rates are reported as missing.

This analysis identifies differences that warrant investigation.
Small subgroup counts limit conclusions about fairness.
The test set remains reserved.

In [21]:
CATBOOST_AUDIT_THRESHOLD = 0.15

catboost_audit_data = pd.DataFrame({
    "Supplier": X_validation["Supplier"].to_numpy(),
    "Actual": np.asarray(y_validation, dtype=int),
    "Predicted": (
        catboost_probability >= CATBOOST_AUDIT_THRESHOLD
    ).astype(int),
})

catboost_supplier_rows = []

for supplier, group in catboost_audit_data.groupby(
    "Supplier", dropna=False
):
    tn, fp, fn, tp = confusion_matrix(
        group["Actual"],
        group["Predicted"],
        labels=[0, 1],
    ).ravel()

    catboost_supplier_rows.append({
        "Supplier": supplier,
        "Orders": len(group),
        "Actual_Non_Compliant": tp + fn,
        "Actual_Compliant": tn + fp,
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "TN": tn,
        "Recall_%": (
            100 * tp / (tp + fn) if tp + fn else np.nan
        ),
        "False_Positive_Rate_%": (
            100 * fp / (fp + tn) if fp + tn else np.nan
        ),
        "Flagged_Orders": tp + fp,
    })

catboost_supplier_audit = (
    pd.DataFrame(catboost_supplier_rows)
    .sort_values("Supplier")
    .reset_index(drop=True)
)

# Verify that supplier counts match the overall confusion matrix.
overall_counts = confusion_matrix(
    catboost_audit_data["Actual"],
    catboost_audit_data["Predicted"],
    labels=[0, 1],
).ravel()

assert np.array_equal(
    catboost_supplier_audit[["TN", "FP", "FN", "TP"]]
    .sum()
    .to_numpy(),
    overall_counts,
)

display(catboost_supplier_audit.round(2))
print("Verification passed: supplier counts match overall results.")

,Supplier,Orders,Actual_Non_Compliant,Actual_Compliant,TP,FN,FP,TN,Recall_%,False_Positive_Rate_%,Flagged_Orders
0,Alpha_Inc,29,3,26,0,3,0,26,0.00,0.00,0
1,Beta_Supplies,34,7,27,6,1,19,8,85.71,70.37,25
2,Delta_Logistics,37,14,23,14,0,21,2,100.00,91.30,35
3,Epsilon_Group,43,1,42,0,1,0,42,0.00,0.00,0
4,Gamma_Co,33,4,29,1,3,8,21,25.00,27.59,9


Verification passed: supplier counts match overall results.


### CatBoost Supplier Audit Findings

At the exploratory threshold of 0.15, CatBoost detects:
- Alpha: 0 of 3 non-compliant orders.
- Beta: 6 of 7.
- Delta: 14 of 14.
- Epsilon: 0 of 1.
- Gamma: 1 of 4.

High detection for Beta and Delta comes with substantial false alarms:
19 of 27 compliant Beta orders and 21 of 23 compliant Delta orders
are incorrectly flagged.

The overall recall of 72.41% conceals substantial supplier differences.
Small positive counts, particularly for Epsilon, limit interpretation.

These results warrant investigation of reliance on supplier identity.
They do not establish discrimination or demonstrate fairness.
The model supports exploratory human review, not automatic supplier
rejection. The test set remains reserved.

In [22]:
supplier_rate_rows = []

for _, row in catboost_supplier_audit.iterrows():
    supplier_rate_rows.extend([
        {
            "Supplier": row["Supplier"],
            "Metric": "Risk detection (recall)",
            "Rate_%": row["Recall_%"],
            "Count": f'{int(row["TP"])}/{int(row["Actual_Non_Compliant"])}',
        },
        {
            "Supplier": row["Supplier"],
            "Metric": "False-positive rate",
            "Rate_%": row["False_Positive_Rate_%"],
            "Count": f'{int(row["FP"])}/{int(row["Actual_Compliant"])}',
        },
    ])

supplier_rate_plot = pd.DataFrame(supplier_rate_rows)

supplier_audit_fig = px.bar(
    supplier_rate_plot,
    x="Supplier",
    y="Rate_%",
    color="Metric",
    barmode="group",
    text="Count",
    hover_data={"Rate_%": ":.2f", "Count": True},
    color_discrete_map={
        "Risk detection (recall)": "#2563EB",
        "False-positive rate": "#F97316",
    },
    title="CatBoost Supplier Audit — Validation, Threshold 0.15",
    labels={"Rate_%": "Rate (%)"},
)

supplier_audit_fig.update_traces(textposition="outside", cliponaxis=False)
supplier_audit_fig.update_yaxes(range=[0, 115], ticksuffix="%")
supplier_audit_fig.update_layout(
    template="plotly_white",
    height=500,
    legend_title_text="",
)

supplier_audit_fig.show()

## 16. Supplier Feature Ablation

We retrain CatBoost without Supplier, retaining Item_Category,
Quantity, Unit_Price, and Negotiated_Price.

We keep the original training settings and chronological split.
The original five-feature model remains available for comparison.

This experiment measures the predictive contribution of including
Supplier under our current setup. Removing Supplier does not
automatically establish fairness, because other inputs may still
reflect supplier differences.

We compare validation ranking and probability error, plus operating
results at the exploratory threshold of 0.15. The same threshold may
produce different review workloads. The test set remains reserved.

In [23]:
NO_SUPPLIER_FEATURES = [
    feature for feature in FEATURES if feature != "Supplier"
]

# Keep the original settings, adjusting the categorical feature list.
no_supplier_params = catboost_model.get_params().copy()
no_supplier_params["cat_features"] = ["Item_Category"]

catboost_no_supplier_model = CatBoostClassifier(
    **no_supplier_params
)

start = perf_counter()

catboost_no_supplier_model.fit(
    X_train[NO_SUPPLIER_FEATURES],
    y_train,
)

print(f"Training completed in {perf_counter() - start:.2f} seconds")

positive_index = list(
    catboost_no_supplier_model.classes_
).index(1)

catboost_no_supplier_probability = (
    catboost_no_supplier_model.predict_proba(
        X_validation[NO_SUPPLIER_FEATURES]
    )[:, positive_index]
)

ablation_rows = []

for experiment, probabilities in {
    "With Supplier — 5 features": catboost_probability,
    "Without Supplier — 4 features": catboost_no_supplier_probability,
}.items():
    prediction = (probabilities >= 0.15).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation, prediction, labels=[0, 1]
    ).ravel()

    ablation_rows.append({
        "Experiment": experiment,
        "ROC_AUC": roc_auc_score(y_validation, probabilities),
        "Average_Precision": average_precision_score(
            y_validation, probabilities
        ),
        "Brier_Score": brier_score_loss(
            y_validation, probabilities
        ),
        "Threshold": 0.15,
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "TN": tn,
        "Precision_%": (
            100 * tp / (tp + fp) if tp + fp else np.nan
        ),
        "Recall_%": (
            100 * tp / (tp + fn) if tp + fn else np.nan
        ),
        "Flagged_Orders": int(prediction.sum()),
    })

supplier_ablation_results = pd.DataFrame(ablation_rows)

display(
    supplier_ablation_results[
        ["Experiment", "ROC_AUC", "Average_Precision", "Brier_Score"]
    ].round(4)
)

display(
    supplier_ablation_results[
        [
            "Experiment", "Threshold", "TP", "FN", "FP", "TN",
            "Precision_%", "Recall_%", "Flagged_Orders",
        ]
    ].round(2)
)

Training completed in 4.01 seconds


,Experiment,ROC_AUC,Average_Precision,Brier_Score
0,With Supplier — 5 features,0.7579,0.3725,0.1233
1,Without Supplier — 4 features,0.5121,0.1846,0.1410


,Experiment,Threshold,TP,FN,FP,TN,Precision_%,Recall_%,Flagged_Orders
0,With Supplier — 5 features,0.15,21,8,48,99,30.43,72.41,69
1,Without Supplier — 4 features,0.15,18,11,78,69,18.75,62.07,96


### Supplier Ablation Findings

Removing Supplier reduced validation ROC-AUC from 0.7579 to 0.5121
and average precision from 0.3725 to 0.1846. Brier score increased
from 0.1233 to 0.1410.

At threshold 0.15, detected non-compliant orders decreased from
21 to 18, while false alarms increased from 48 to 78.
Orders flagged for review increased from 69 to 96.

Supplier identity provides substantial predictive value under the
current dataset, model settings, and chronological split.

This does not establish causation or fairness, and does not demonstrate
generalization to unseen suppliers. Removing Supplier did not improve
overall validation performance.

We retain the original five-feature CatBoost as our provisional
candidate. Its supplier-level limitations remain documented.
The test set remains reserved.

## 17. Frozen Model and Threshold for Test Evaluation

For this portfolio demonstration, we assume a review capacity of up to
40% of orders. This illustrative assumption was chosen after exploring
validation results. It is not a confirmed business requirement or a
pre-specified evaluation criterion.

We select the original five-feature CatBoost at threshold 0.15.
Among the evaluated model/threshold combinations, it achieves the
highest validation recall within this illustrative workload limit.

Validation results:
- Detected non-compliant orders: 21 of 29 (72.41% recall).
- Orders flagged: 69 of 176 (39.20% workload).
- False alarms: 48.
- Missed non-compliant orders: 8.

We freeze:
- Model: original CatBoost with 300 iterations, depth 4,
  learning rate 0.03, and random seed 42.
- Inputs: Supplier, Item_Category, Quantity, Unit_Price,
  and Negotiated_Price.
- Threshold: 0.15.
- Training data: the original 396 training orders.
- No refitting on validation data before this test evaluation.

The 40% limit applies to observed validation workload. A fixed threshold
does not guarantee the same workload on future orders or the test set.

We will evaluate the frozen model on the chronological test set and
report the results without adjusting our choices in response to them.
Earlier descriptive exploration included the full dataset, so this
is a reserved modelling test set rather than completely unseen data.

Supplier-level limitations and undocumented compliance-label timing
remain constraints on operational use.

In [24]:
# Record the decision without evaluating the test set.
FROZEN_MODEL = catboost_model
FROZEN_FEATURES = FEATURES.copy()
FROZEN_THRESHOLD = 0.15

print("Frozen model: original five-feature CatBoost")
print("Features:", FROZEN_FEATURES)
print("Threshold:", FROZEN_THRESHOLD)
print("Training orders:", len(X_train))
print("Test evaluation has not run in this cell.")

Frozen model: original five-feature CatBoost
Features: ['Supplier', 'Item_Category', 'Quantity', 'Unit_Price', 'Negotiated_Price']
Threshold: 0.15
Training orders: 396
Test evaluation has not run in this cell.
